# Model Inspection and Sanity Checks
<!-- structured-notebook -->
## Notebook Summary
Purpose: small utilities for validating a trained BERTopic model after a tuning round — confirming topic counts match expectations, probing the outlier (−1) topic, (re-)saving the topic-to-document mapping, and re-attaching timestamps when they were lost.

Main steps:
- Run sanity checks comparing saved artifacts against the model's own `get_topic_info()`.
- Inspect the outlier topic to understand what the model considered noise.
- Re-save the doc-topic mapping as a pickle (useful when the original save step was missed).
- Re-attach `created_utc` timestamps to topic assignments if the original timestamping run failed.


# Sanity Checks
<!-- structured-notebook -->
## Summary
Quick assertions: topic counts in `topics.csv` match `model.get_topic_info()`, probability matrix shape matches number of unique docs, hard topic assignments live in the expected range (`-1..n_topics-1`).


In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_OUTPUT,
)
import numpy as np, pandas as pd, pickle, os

round_dir = "round_10"    # where you wrote parquet
r4 = "round_4"

df_raw = pd.read_json(REDDIT_OUTPUT / "merged_submissions.jsonl", lines=True).reset_index(drop=True)

kept_idx = np.load(f"{r4}/kept_indices.npy")                # K?
map_o2u  = np.load(f"{r4}/map_orig_to_unique.npy")          # K?
ts_K     = np.load(f"{r4}/kept_ts_seconds.npy")             # K?
topics_K = np.load(f"{round_dir}/train_topics_original.npy")# K?

print("K sizes:", kept_idx.shape, map_o2u.shape, ts_K.shape, topics_K.shape)
print("K (expected rows in parquet):", kept_idx.shape[0])
print("non_noise_count:", int((topics_K>=0).sum()))

# what did you actually write?
meta = pd.read_parquet(f"{round_dir}/kept_metadata.parquet")
print("parquet rows:", meta.shape[0])

# sanity: was a filter applied?
print("min/max topic_hard in parquet:", meta["topic_hard"].min(), meta["topic_hard"].max())
print("rows with topic_hard>=0:", int((meta["topic_hard"]>=0).sum()))

# Outlier Topic Inspection
<!-- structured-notebook -->
## Summary
Topic −1 in BERTopic is the "outlier" topic — documents HDBSCAN did not confidently assign to any cluster. Inspecting what ended up there is a useful quality check: a healthy model has noise-like content there; a poorly-tuned model dumps meaningful topics into −1.


In [ ]:
from bertopic import BERTopic

topic_model = BERTopic.load("round_2/bertopic_final.model")

# Check all topics
topics = topic_model.get_topics()

counts = topic_model.get_topic_info()["Count"].values
print("Counts of topics:", counts)


# Re-Save Topic-Doc Mapping
<!-- structured-notebook -->
## Summary
The main training script saves a per-document topic assignment, but in early rounds this was sometimes skipped. This utility reconstructs the mapping from a trained model + the preprocessed corpus and writes it as a pickle.


In [ ]:
from collections import defaultdict, Counter
import pickle
import numpy as np

# Load preprocessed docs and the *training-time* labels
with open("round_2/preprocessed-docs.pkl", "rb") as f:
    all_processed_docs = pickle.load(f)

with open("round_2/train_topics.npy", "rb") as f:
    topics = np.load(f)

assert len(all_processed_docs) == len(topics), "Docs and topics length mismatch!"

# Build mapping from fixed labels
topic_doc_map = defaultdict(list)
for doc, topic_id in zip(all_processed_docs, topics):
    topic_doc_map[int(topic_id)].append(doc)

# Save
topic_doc_map_path = "round_2/topic_doc_map.pkl"
with open(topic_doc_map_path, "wb") as f:
    pickle.dump(topic_doc_map, f)

print(f"Saved topic-to-doc mapping to {topic_doc_map_path}")

# Optional sanity check (counts should match your training summary)
from bertopic import BERTopic
topic_model = BERTopic.load("round_2/bertopic_final.model")
df = topic_model.get_topic_info()
train_counts = df.set_index("Topic")["Count"].to_dict()
map_counts = Counter(map(int, topics))
print("Train vs Map count for topic 1:", train_counts.get(1), map_counts.get(1))

# Re-Attach Timestamps
<!-- structured-notebook -->
## Summary
Parallel to the main `02_timestamp_attachment` notebook. This is the original script version; use the notebook for new runs. Kept here for comparison and for cases where a scripted batch run is preferred.


In [ ]:
# ======= timestamp_attach_no_chunks.py =======
import copy
import os
import re
import pickle
from collections import defaultdict, deque
from typing import Optional, Callable, Dict, List

import numpy as np
import pandas as pd
from tqdm import tqdm
import emoji
import spacy
from langdetect import detect, DetectorFactory

# Deterministic language detection like before
DetectorFactory.seed = 0

# ---- Your exact preprocessing (single-row mirror) ----
# Load spaCy once
_nlp = None
def _get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    return _nlp

def single_preprocess_fn(text: str) -> Optional[str]:
    """Exact mirror of your pipeline for a SINGLE string."""
    if not isinstance(text, str) or len(text) < 5:
        return None

    # clean_text(...) part
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"u/\w+|r/\w+|>\s.*", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower().strip()

    # Language gate
    try:
        if detect(text) != "en":
            return None
    except Exception:
        return None

    if len(text.split()) <= 3:
        return None

    # Lemmatize + stopwords + alpha only
    nlp = _get_nlp()
    doc = nlp(text)
    tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
    if not tokens:
        return None

    return " ".join(tokens)


# ---- Builder: cleaned_text → deque[timestamps] ----
def build_text_to_ts_index(
    raw_path: str,
    title_col: str = "title",
    body_col: str = "selftext",
    ts_col: str = "created_utc",
    single_preprocess: Callable[[str], Optional[str]] = single_preprocess_fn,
    assume_created_utc_seconds: bool = True
) -> Dict[str, deque]:
    """
    Load raw file (CSV/JSON/JSONL/Parquet), preprocess each (title+body),
    and return dict: cleaned_text -> deque([timestamps...]).
    """
    # Load raw once (75k rows is fine)
    if raw_path.endswith(".csv"):
        df = pd.read_csv(raw_path)
    elif raw_path.endswith(".parquet"):
        df = pd.read_parquet(raw_path)
    elif raw_path.endswith(".json") or raw_path.endswith(".jsonl"):
        df = pd.read_json(raw_path, lines=True)
    else:
        raise ValueError("Unsupported raw file format. Use CSV/Parquet/JSON/JSONL.")

    # Validate columns
    for col in (title_col, body_col, ts_col):
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in raw file.")

    # Normalize timestamp
    if assume_created_utc_seconds and ts_col == "created_utc":
        df["_ts"] = pd.to_datetime(df[ts_col], unit="s", utc=True, errors="coerce")
    else:
        df["_ts"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")

    df = df.fillna({title_col: "", body_col: ""}).astype({title_col: str, body_col: str})
    texts = (df[title_col] + " " + df[body_col]).tolist()
    tss = df["_ts"].tolist()

    text2ts: Dict[str, deque] = defaultdict(deque)
    kept = 0
    for raw_txt, ts in tqdm(zip(texts, tss), total=len(texts), desc="Indexing timestamps"):
        cleaned = single_preprocess(raw_txt)
        if cleaned:
            text2ts[cleaned].append(ts)
            kept += 1

    print(f"Raw rows: {len(df)} | kept after preprocessing: {kept}")
    print(f"Unique cleaned strings indexed: {len(text2ts)}")
    return text2ts


# ---- Sanity check (dry run) + attach ----
def attach_timestamps_to_topic_map(
    topic_doc_map_path: str,
    text2ts: Dict[str, deque],
    out_path_pkl: str,
    out_path_csv: str,
    dry_run: bool = True,
    preview_n: int = 10
):
    """
    Attach timestamps to existing topic→docs mapping using text2ts index.
    If dry_run=True, do NOT write files; instead print a report and samples.
    """
    with open(topic_doc_map_path, "rb") as f:
        topic_doc_map = pickle.load(f)

    total_docs = 0
    matched = 0
    per_topic = []

    # Build augmented mapping (in-memory)
    topic_doc_map_with_ts: Dict[int, List[dict]] = {}
    for k in topic_doc_map.keys():
        try:
            topic_id = int(k)  # keys might be np.int64
        except Exception:
            topic_id = k

        docs = topic_doc_map[k]
        aug = []
        topic_matched = 0

        for doc in docs:
            total_docs += 1
            ts = None
            q = text2ts.get(doc)
            if q and len(q) > 0:
                ts = q.popleft()
                matched += 1
                topic_matched += 1
            aug.append({"text": doc, "timestamp": ts})
        topic_doc_map_with_ts[topic_id] = aug
        per_topic.append({
            "topic": topic_id,
            "docs": len(docs),
            "matched": topic_matched,
            "match_rate": topic_matched / max(1, len(docs))
        })

    df_report = pd.DataFrame(per_topic).sort_values("match_rate")
    overall_rate = matched / max(1, total_docs)
    print("\n=== DRY RUN REPORT ===" if dry_run else "\n=== ATTACH REPORT ===")
    print(f"Total docs: {total_docs} | Matched: {matched} | Match rate: {overall_rate:.2%}")
    print("Worst-match topics (bottom 10 by match_rate):")
    print(df_report.head(10).to_string(index=False))

    # Sample a few rows to inspect
    sample_rows = []
    for t, rows in topic_doc_map_with_ts.items():
        for r in rows[:2]:  # first 2 per topic
            sample_rows.append({"topic": t, "text": r["text"][:80], "timestamp": r["timestamp"]})
        if len(sample_rows) >= preview_n:
            break
    print("\nSample rows:")
    print(pd.DataFrame(sample_rows))

    if dry_run:
        print("\nDry run complete. No files were written.")
        return df_report, topic_doc_map_with_ts  # so you can inspect programmatically

    # Write outputs
    os.makedirs(os.path.dirname(out_path_pkl), exist_ok=True)
    with open(out_path_pkl, "wb") as f:
        pickle.dump(topic_doc_map_with_ts, f)
    print(f"Saved: {out_path_pkl}")

    # Long-form CSV for timeline work
    rows_long = []
    for t, rows in topic_doc_map_with_ts.items():
        for r in rows:
            rows_long.append({"topic": t, "text": r["text"], "timestamp": r["timestamp"]})
    df_long = pd.DataFrame(rows_long)
    df_long.to_csv(out_path_csv, index=False)
    print(f"Saved: {out_path_csv}")

    return df_report, topic_doc_map_with_ts

# 1) Build cleaned_text → timestamps index from your raw file
text2ts = build_text_to_ts_index(
    raw_path=REDDIT_OUTPUT / "merged_submissions.jsonl",  # or .csv/.parquet
    title_col="title",
    body_col="selftext",
    ts_col="created_utc",          # adjust if different
    assume_created_utc_seconds=True # if created_utc is seconds
)

# 2) Dry-run (NO files written) to inspect match rates + samples
report, preview_map = attach_timestamps_to_topic_map(
    topic_doc_map_path="../topic_modelling_v2/round_2/topic_doc_map.pkl",
    text2ts=copy.deepcopy(text2ts),
    out_path_pkl="../topic_modelling_v2/round_2/topic_doc_map_with_ts.pkl",
    out_path_csv="../topic_modelling_v2/round_2/topic_doc_with_ts_long.csv",
    dry_run=True,           # <- stays True for sanity check
    preview_n=12
)

# Inspect `report` and the printed samples. If match rate looks good:

# 3) Commit (write files)
report, final_map = attach_timestamps_to_topic_map(
    topic_doc_map_path="../topic_modelling_v2/round_2/topic_doc_map.pkl",
    text2ts=text2ts,
    out_path_pkl="../topic_modelling_v2/round_2/topic_doc_map_with_ts.pkl",
    out_path_csv="../topic_modelling_v2/round_2/topic_doc_with_ts_long.csv",
    dry_run=False,          # <- now we write
    preview_n=0
)